# paper-verdict-finetune -- Colab GPU runner

Before running: Runtime -> Change runtime type -> GPU.

This notebook is intentionally thin: it clones the repo, installs
requirements, and calls the scripts in `src/`. All logic lives there.

**Persistence model** (so a run can be launched, closed, and reviewed
later -- Colab's local disk does not survive a runtime reset):
- **Checkpoints + full per-run outputs** -> copied to Google Drive,
  under `MyDrive/paper-verdict-finetune/runs/<run_id>/`. Weights are too
  large to put in git on every run.
- **Lightweight run record** (config, metrics, confusion matrix -- no
  weights) -> committed and pushed back to this branch, so `RUNS.md` in
  the repo is always a current, append-only index of every run, readable
  from any future session without mounting Drive or downloading
  anything. Drive holds the heavy artifact; GitHub holds the map of what
  exists and what it scored.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_RUNS_DIR = '/content/drive/MyDrive/paper-verdict-finetune/runs'
os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)

In [ ]:
BRANCH = "verdict-finetune-architecture"
!git clone --branch $BRANCH --depth 1 https://github.com/AngryWatership/textor.git repo
%cd repo
!git config user.email "noreply@anthropic.com"
!git config user.name "Colab runner"

In [ ]:
!pip install -q -r requirements.txt

## Review past runs before launching a new one

`RUNS.md` came in with the clone -- it's the current index of everything
already tried. Look at this before picking the next config, so you're
not re-running something already done, and so you know what the current
best is to try to beat.

In [ ]:
print(open("RUNS.md").read())

In [ ]:
# sanity-check the split before committing GPU time to training
!python src/data.py --data data/papers.jsonl --split-out outputs/split.json

In [ ]:
import sys
sys.path.insert(0, "src")
from runlog import new_run_id

BASE_MODEL = "allenai/specter2_base"  # or sentence-transformers/all-MiniLM-L6-v2, allenai/scibert_scivocab_uncased
FINETUNE_MODE = "partial"  # or "full"

RUN_ID = new_run_id(BASE_MODEL, FINETUNE_MODE)
print("RUN_ID:", RUN_ID)
os.environ["RUN_ID"] = RUN_ID
os.environ["BASE_MODEL"] = BASE_MODEL
os.environ["FINETUNE_MODE"] = FINETUNE_MODE

In [ ]:
!cd src && python train.py \
    --base-model $BASE_MODEL \
    --finetune-mode $FINETUNE_MODE \
    --run-id $RUN_ID \
    --data ../data/papers.jsonl \
    --split-out ../outputs/split.json \
    --out-dir ../outputs \
    --repo-root ..

In [ ]:
!cd src && python evaluate.py \
    --run-id $RUN_ID \
    --base-model $BASE_MODEL \
    --finetune-mode $FINETUNE_MODE \
    --data ../data/papers.jsonl \
    --split-out ../outputs/split.json \
    --out-dir ../outputs \
    --repo-root ..

In [ ]:
import json
report = json.load(open(f"outputs/{RUN_ID}/eval_report.json"))
print("confusion matrix (rows=true, cols=pred), labels:", report["confusion_matrix_labels"])
for row in report["confusion_matrix"]:
    print(row)
print("test silhouette:", report["test_silhouette_euclidean"])
print("test knn10 purity:", report["test_knn10_purity"])
print("per-class purity:", report["test_knn10_purity_per_class"])
print("\ncompare against docs/prior_results.md and RUNS.md for every other run so far.")

## Persist: full run (incl. weights) to Drive, lightweight record to GitHub

In [ ]:
import shutil
drive_run_dir = f"{DRIVE_RUNS_DIR}/{RUN_ID}"
shutil.copytree(f"outputs/{RUN_ID}", drive_run_dir, dirs_exist_ok=True)
print("copied full run (including best_model.pt) to", drive_run_dir)

# record the Drive path in the log, then regenerate RUNS.md with it, so a
# later session knows exactly where to find the checkpoint without having
# downloaded it
from runlog import upsert_run
upsert_run(".", RUN_ID, {"checkpoint_path": f"{drive_run_dir}/best_model.pt"})

In [ ]:
# push the lightweight record (RUNS.md, runs_log.jsonl, this run's config/
# train_summary/eval_report -- NOT best_model.pt, that only lives on Drive)
# back to the branch, so it's reviewable from any future session without
# Colab or Drive at all.
!git add RUNS.md runs_log.jsonl outputs/$RUN_ID/config.json outputs/$RUN_ID/train_summary.json outputs/$RUN_ID/eval_report.json
!git commit -m "Log run $RUN_ID" || echo "nothing new to commit"
!git push origin $BRANCH

## To resume / review in a later session

- `git clone` this branch -> `RUNS.md` shows every run and its metrics,
  no Drive or Colab needed to see *what happened*.
- To re-load an actual checkpoint for further use, mount Drive and point
  `evaluate.py --checkpoint` at
  `MyDrive/paper-verdict-finetune/runs/<run_id>/best_model.pt`.